In [ ]:
from libs import circuits, dataloading, qnn_models
import matplotlib.pyplot as plt
plt.style.use('./notebookstyle_aps.mplstyle')

import joblib
from joblib import cpu_count, Parallel, delayed
import numpy as np
import json
from copy import deepcopy
import jax
import pennylane as qml
import csv
import json
import os
import pandas as pd


%load_ext autoreload
%autoreload 2

## random keys for initialization

In [ ]:
key = jax.random.PRNGKey(12345)
keys = jax.random.split(key, 10)

## data loading

In [ ]:
save_dir = "./QNN_results/"

In [ ]:
jnp_train_inputs, jnp_val_inputs, jnp_train_outputs, jnp_val_outputs = dataloading.load_data(0, n_samples = 'all')

## circuit example

In [ ]:
n_enc = 2; n_dec = 2; n_qubits = 2; local_observable = 1

mycircuit = circuits.PQCircuit(n_enc, n_dec, n_qubits, var_angles = 'XYZ', encoding_angles='XY', choose_freq = True)
    
dev = qml.device("default.qubit", wires=n_qubits)

model_qnn, no_params, model_name = qnn_models.make_model(dev, mycircuit, local_observable)

from functools import reduce
circuit = mycircuit
if local_observable:
        observable_indices = list(range(circuit.n_wires))
        obs_return = lambda: [qml.expval(qml.PauliZ(i)) for i in observable_indices]
else:
    full_obs = reduce(lambda x, y: x @ y, [qml.PauliZ(i) for i in range(circuit.n_wires)])
    obs_return = lambda: [qml.expval(full_obs)]
        
@qml.qnode(dev, interface="jax")
def qnn_pqc(inputs, angles):
    circuit(inputs, angles,  dev.wires)
    return obs_return()
        
pars = np.hstack([np.ones(n_enc*6)*0.5, np.ones(no_params - n_enc*6)*0.3])
print(qml.draw(qnn_pqc, level = 'device')(np.arange(6)[None,:]*2, pars))

## functions for training and to check which results are missing

In [ ]:
def make_train_eval(key, n_qubits, n_enc, n_dec, local_observable):
    
    mycircuit = circuits.PQCircuit(n_enc, n_dec, n_qubits, var_angles = 'XYZ', encoding_angles='XYZ', choose_freq = True)
    
    dev = qml.device("default.qubit", wires=n_qubits)
    
    model_qnn, no_params, model_name = qnn_models.make_model(dev, mycircuit, local_observable)
    
    metrics = qnn_models.train_and_evaluate(model_qnn, model_name, no_params,
                                        key, jnp_train_inputs, jnp_train_outputs, jnp_val_inputs, jnp_val_outputs, 
                                        save_dir,
                                        save_postfix = str(key),
                                        n_epochs = 150, learning_rate = 0.0005)

    return metrics

In [ ]:
dfq = pd.read_csv('./QNN_results/QNN_r2_noduplicates.txt', header =0, index_col = 0)

import re
def parse_config(s):
    match = re.match(r"(\d+)qubits_circuit_([a-zA-Z]+)_([a-zA-Z]+)_freq(\d+)enc_(\d+)dec_([a-zA-Z]+)", s)
    if match:
        qubits, enc_angles, angles, n_enc, n_dec,  entanglement = match.groups()
        return int(qubits), int(n_enc), int(n_dec), entanglement, angles, enc_angles
    return None, None, None, None

dfq[['n_qubits', 'n_enc', 'n_dec', 'entanglement','angles','enc_angles']] = dfq['model_name'].apply(
    lambda x: pd.Series(parse_config(x))
)

dfq['nonlocal_flag'] = (dfq.entanglement == 'nonlocal')
dfq['n_params'] = dfq.n_qubits*(dfq.n_enc + dfq.n_dec)*dfq.angles.apply(lambda s: len(s)) + dfq.n_qubits*dfq.n_enc*dfq.enc_angles.apply(lambda s: len(s))

### training for a given number of qubits

In [ ]:
dfq = dfq.reset_index()
key_map = {str(k): i for i, k in enumerate(keys)}
dfq["key_id"] = dfq["key"].astype(str).map(key_map)

In [ ]:
# select number of qubits
df1 = dfq[dfq.n_qubits == 2]

# hyperparameter and initialization combinations
existing = {
    (r.key_id, r.n_enc, r.n_dec, r.nonlocal_flag)
    for r in df1[['key_id','n_enc','n_dec','nonlocal_flag']].itertuples(index=False)
}

# collect jobs for training with different hyperparameters and initializations
jobs = []
for n_dec in range(0,7):
    for n_enc in range(1, 10):
        for local_observable in [0, 1]:
            for key_id,key in enumerate(keys):
                combo = (key_id, n_enc, n_dec, abs(local_observable-1))
                if combo not in existing:
                    print("Running:", n_enc, n_dec, local_observable)
                    jobs.append((key, n_enc, n_dec, local_observable))
                else:
                    print("Skipping:", key_id, n_enc, n_dec, local_observable, 'already exists')

def run(job):
    key, n_enc, n_dec, local_observable = job
    return make_train_eval(key, 2, n_enc, n_dec, local_observable)

# run in parallel
n_jobs = 6
results = Parallel(n_jobs=n_jobs, backend="loky", batch_size=1)(
    delayed(run)(job) for job in jobs
)
